In [1]:
# Importações
import pandas as pd
from pandas.plotting import register_matplotlib_converters

register_matplotlib_converters()

import pytz
import MetaTrader5 as mt5

import sys
from pathlib import Path

from datetime import datetime

# Adiciona a pasta 'src' ao path para permitir as importações dos nossos módulos
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

ticker = 'WDO$'


In [2]:
# conecte-se ao MetaTrader 5
if not mt5.initialize():
    print("initialize() failed")
    mt5.shutdown()
 
# consultamos o estado e os parâmetros de conexão
print(mt5.terminal_info())
# obtemos informações sobre a versão do MetaTrader 5
print(mt5.version())

TerminalInfo(community_account=True, community_connection=True, connected=True, dlls_allowed=True, trade_allowed=False, tradeapi_disabled=False, email_enabled=False, ftp_enabled=False, notifications_enabled=False, mqid=False, build=5327, maxbars=100000, codepage=1252, ping_last=19732, community_balance=0.0, retransmission=0.7134561141529783, company='Clear (XP Investimentos CCTVM)', name='Clear Investimentos MT5 Terminal', language='Portuguese (Brazil)', path='C:\\Program Files\\Clear Investimentos MT5 Terminal', data_path='C:\\Users\\User\\AppData\\Roaming\\MetaQuotes\\Terminal\\698B86206820B42976F30D28CAC50412', commondata_path='C:\\Users\\User\\AppData\\Roaming\\MetaQuotes\\Terminal\\Common')
(500, 5327, '3 Oct 2025')


In [4]:
# obtemos o número de instrumentos financeiros
symbols=mt5.symbols_total()
if symbols>0:
    print("Total symbols =",symbols)
else:
    print("Symbols not found")

Total symbols = 62398


In [ ]:
symbol_info = mt5.symbol_info(ticker)
if symbol_info is None:
    print(f"{ticker} not found, can not call order_check()")
    mt5.shutdown()

print(symbol_info)

trade_type_buy = mt5.ORDER_TYPE_BUY 
trade_type_sell = mt5.ORDER_TYPE_SELL
print("trade_type_buy:", trade_type_buy)
print("trade_type_sell:", trade_type_sell)

price_buy = symbol_info.ask
price_sell = symbol_info.bid
print("price_buy:", price_buy)
print("price_sell:", price_sell)



SymbolInfo(custom=False, chart_mode=1, select=True, visible=True, session_deals=0, session_buy_orders=0, session_sell_orders=0, volume=1134, volumehigh=77662796314, volumelow=147, time=1760120880, digits=3, spread=0, spread_float=True, ticks_bookdepth=32, trade_calc_mode=33, trade_mode=0, start_time=0, expiration_time=0, trade_stops_level=0, trade_freeze_level=0, trade_exemode=3, swap_mode=0, swap_rollover3days=3, margin_hedged_use_leg=False, expiration_mode=2, filling_mode=3, order_mode=127, order_gtc_mode=2, option_mode=0, option_right=0, bid=0.0, bidhigh=0.0, bidlow=0.0, ask=0.0, askhigh=0.0, asklow=0.0, last=5555.5, lasthigh=5560.0, lastlow=5387.5, volume_real=1134.0, volumehigh_real=77662796314.52242, volumelow_real=147.0, option_strike=0.0, point=0.001, trade_tick_value=0.01, trade_tick_value_profit=0.01, trade_tick_value_loss=0.01, trade_tick_size=0.001, trade_contract_size=1.0, trade_accrued_interest=0.0, trade_face_value=0.0, trade_liquidity_rate=0.0, volume_min=1.0, volume_ma

In [6]:
ticks = mt5.copy_ticks_range(ticker, datetime(2025, 10, 9, 0, 0), 1000, mt5.COPY_TICKS_ALL)

In [11]:
selected = mt5.symbol_select(ticker,True)

if not selected:
    print(f"Failed to select {ticker}")
    mt5.shutdown()
    quit()

# imprimimos o último tick do símbolo WDO$
lasttick=mt5.symbol_info_tick(ticker)
print(f"Ultimo Tick: {lasttick}")

timezone = pytz.timezone("Etc/UTC")

print(f"Show symbol_info_tick(\"{ticker}\")._asdict():")
symbol_info_tick_dict = mt5.symbol_info_tick(ticker)._asdict()
# converte o time em formato legível
symbol_info_tick_dict["time"] = datetime.fromtimestamp(symbol_info_tick_dict["time"], timezone)
symbol_info_tick_dict["time_msc"] = datetime.fromtimestamp(symbol_info_tick_dict["time_msc"]/1000, timezone)
# imprime todos os elementos de symbol_info_tick_dict
for prop in symbol_info_tick_dict:
    print("  {}={}".format(prop, symbol_info_tick_dict[prop]))
 
# concluímos a conexão ao terminal MetaTrader 5
#mt5.shutdown()

Ultimo Tick: Tick(time=1760120880, bid=0.0, ask=0.0, last=5555.5, volume=1134, time_msc=1760120880000, flags=24, volume_real=1134.0)
Show symbol_info_tick("WDO$")._asdict():
  time=2025-10-10 18:28:00+00:00
  bid=0.0
  ask=0.0
  last=5555.5
  volume=1134
  time_msc=2025-10-10 18:28:00+00:00
  flags=24
  volume_real=1134.0


In [5]:
# obtemos informações sobre um instrumento financeiro

timeframe = mt5.TIMEFRAME_H1
print(f"Utilizando timeframe: {timeframe}")

range = mt5.copy_rates_range(ticker, mt5.TIMEFRAME_M5, datetime(2025,10,1,0), datetime(2025,10,11,0))
df_range = pd.DataFrame(range)
df_range['time']=pd.to_datetime(df_range['time'], unit='s')
print(df_range)

print(f"Quantidade de candles {df_range.shape}")

Utilizando timeframe: 16385
                   time    open    high     low   close  tick_volume  spread  \
0   2025-10-01 09:00:00  5347.5  5349.5  5342.0  5343.5        17963       1   
1   2025-10-01 09:05:00  5343.5  5352.0  5343.5  5350.5        14521       1   
2   2025-10-01 09:10:00  5350.5  5354.5  5349.5  5351.5        10058       1   
3   2025-10-01 09:15:00  5351.5  5352.5  5334.0  5336.5        28165       1   
4   2025-10-01 09:20:00  5337.0  5342.5  5334.5  5341.5        17211       1   
..                  ...     ...     ...     ...     ...          ...     ...   
907 2025-10-10 18:05:00  5545.0  5547.0  5543.0  5545.5          470       0   
908 2025-10-10 18:10:00  5546.0  5553.0  5545.0  5551.5         1116       0   
909 2025-10-10 18:15:00  5551.0  5560.0  5549.0  5550.5         1525       0   
910 2025-10-10 18:20:00  5550.0  5555.0  5550.0  5550.5          704       0   
911 2025-10-10 18:25:00  5550.5  5560.0  5550.0  5559.5          841       0   

     real_v

In [3]:
# obtemos informações sobre um instrumento financeiro

timeframe = mt5.TIMEFRAME_M5
#print(f"Utilizando timeframe: {timeframe}")

#candle_position = 114 if datetime.now().time() > datetime.strptime("18:30", "%H:%M").time() else 0
#print(f"Utilizando candle_position: {candle_position}")

range = mt5.copy_rates_from_pos("WDO$", timeframe, 562, 300)
df_range = pd.DataFrame(range)
df_range['time']=pd.to_datetime(df_range['time'], unit='s')
print(df_range)

                   time    open    high     low   close  tick_volume  spread  \
0   2025-10-01 13:10:00  5370.5  5372.5  5369.5  5371.5         3872       1   
1   2025-10-01 13:15:00  5371.0  5372.0  5369.5  5371.0         2111       1   
2   2025-10-01 13:20:00  5371.5  5372.5  5371.0  5371.5         1527       1   
3   2025-10-01 13:25:00  5371.0  5372.0  5369.0  5369.5         2519       1   
4   2025-10-01 13:30:00  5369.5  5371.0  5369.0  5369.0         2113       1   
..                  ...     ...     ...     ...     ...          ...     ...   
295 2025-10-06 09:15:00  5366.0  5367.5  5363.0  5365.5        11426       1   
296 2025-10-06 09:20:00  5365.5  5368.0  5363.5  5367.5         9066       1   
297 2025-10-06 09:25:00  5367.5  5368.0  5363.5  5365.5         6079       1   
298 2025-10-06 09:30:00  5365.0  5371.5  5364.5  5371.0        12169       1   
299 2025-10-06 09:35:00  5371.0  5371.5  5366.5  5367.0         7174       1   

     real_volume  
0          20804  
1

In [ ]:
# obtemos ticks de um instrumento financeiro

ticks = mt5.copy_ticks_from("WIN$", datetime(2020,1,1,0), 1000, mt5.COPY_TICKS_ALL)
df_ticks = pd.DataFrame(ticks)
print(df_ticks)

Empty DataFrame
Columns: []
Index: []


In [ ]:
mt5.copy_rates_from_pos(ticker, timeframe, candle_position, count)

In [17]:
# desligamos a conexão com o MetaTrader 5
mt5.shutdown()

True